In [28]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [29]:
df_train_feature = pd.read_csv("../data/train_feature_selected.csv")
df_train_feature.head()


,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h,area_first_ha,log1p_area_first,log1p_growth,log_area_ratio_0_5h,centroid_speed_m_per_h,spread_bearing_deg,dist_min_ci_0_5h,...,dist_fit_r2_0_5h,alignment_abs,cross_track_component,event_start_hour,event_start_dayofweek,event_start_month,time_to_hit_hours,event,distance_speed,fire_pressure
0,3,4.265188,0,79.696304,4.390693,1.354787,0.03545,1.940119,70.130507,6166.121596,...,0.886373,0.054649,-1.937219,19,4,5,18.892512,0,76.411450,0.012923
1,2,1.169918,0,8.946749,2.297246,0.000000,0.00000,0.000000,0.000000,2930.925956,...,0.000000,0.568898,-0.000000,4,4,6,22.048108,1,294.661687,0.003051
2,4,4.777526,0,106.482638,4.677329,0.000000,0.00000,0.000000,0.000000,3272.375090,...,0.000000,0.882385,0.000000,22,4,8,0.888895,1,30.445616,0.032530
3,1,0.000000,1,67.631125,4.228746,0.000000,0.00000,0.000000,0.000000,64119.871377,...,0.000000,0.000000,0.000000,20,5,8,60.953021,0,934.268113,0.001055
4,2,4.975273,0,35.632874,3.600946,0.000000,0.00000,0.000000,0.000000,18005.432261,...,0.000000,0.934634,-0.000000,21,5,7,44.990274,0,491.510235,0.001979


In [30]:
df_train_feature.columns

Index(['num_perimeters_0_5h', 'dt_first_last_0_5h',
       'low_temporal_resolution_0_5h', 'area_first_ha', 'log1p_area_first',
       'log1p_growth', 'log_area_ratio_0_5h', 'centroid_speed_m_per_h',
       'spread_bearing_deg', 'dist_min_ci_0_5h', 'dist_accel_m_per_h2',
       'dist_fit_r2_0_5h', 'alignment_abs', 'cross_track_component',
       'event_start_hour', 'event_start_dayofweek', 'event_start_month',
       'time_to_hit_hours', 'event', 'distance_speed', 'fire_pressure'],
      dtype='object')

# Feature Type Grouping (Final Dataset)

In [31]:
# ================================
# FEATURE GROUPING
# ================================

# Target columns
TARGET_TIME = "time_to_hit_hours"
TARGET_EVENT = "event"

# ID column (if present)
ID_COL = "event_id"

# ----------------
# Continuous features
# ----------------

continuous = [

"area_first_ha",
"log1p_area_first",
"log1p_growth",
"log_area_ratio_0_5h",

"centroid_speed_m_per_h",

"dist_min_ci_0_5h",
"dist_accel_m_per_h2",
"dist_fit_r2_0_5h",

"distance_speed",
"fire_pressure",

"cross_track_component"

]

# ----------------
# Direction / Geometry features
# ----------------

direction_features = [

"spread_bearing_deg",
"alignment_abs"

]

# ----------------
# Temporal features
# ----------------

time_features = [

"event_start_hour",
"event_start_dayofweek",
"event_start_month"

]

# ----------------
# Observation features
# ----------------

observation_features = [

"num_perimeters_0_5h",
"dt_first_last_0_5h",
"low_temporal_resolution_0_5h"

]

# ----------------
# All features
# ----------------

FEATURES = (
continuous
+ direction_features
+ time_features
+ observation_features
)

print("Continuous:", len(continuous))
print("Direction:", len(direction_features))
print("Time:", len(time_features))
print("Observation:", len(observation_features))

print("\nTotal Features:", len(FEATURES))
print(FEATURES)

Continuous: 11
Direction: 2
Time: 3
Observation: 3

Total Features: 19
['area_first_ha', 'log1p_area_first', 'log1p_growth', 'log_area_ratio_0_5h', 'centroid_speed_m_per_h', 'dist_min_ci_0_5h', 'dist_accel_m_per_h2', 'dist_fit_r2_0_5h', 'distance_speed', 'fire_pressure', 'cross_track_component', 'spread_bearing_deg', 'alignment_abs', 'event_start_hour', 'event_start_dayofweek', 'event_start_month', 'num_perimeters_0_5h', 'dt_first_last_0_5h', 'low_temporal_resolution_0_5h']


In [32]:
df_train_feature[continuous].skew()

area_first_ha             4.721167
log1p_area_first         -0.111519
log1p_growth              3.635148
log_area_ratio_0_5h       6.337715
centroid_speed_m_per_h    6.959181
dist_min_ci_0_5h          1.634553
dist_accel_m_per_h2      -8.269464
dist_fit_r2_0_5h          3.884857
distance_speed            7.843248
fire_pressure             3.674657
cross_track_component     2.425414
dtype: float64

In [33]:
from sklearn.preprocessing import PowerTransformer

pt = PowerTransformer(method="yeo-johnson")

df_train_feature["dist_accel_m_per_h2"] = pt.fit_transform(
    df_train_feature[["dist_accel_m_per_h2"]]
)

In [34]:

df_train_feature[continuous].skew().sort_values()

log1p_area_first         -0.111519
dist_accel_m_per_h2       0.848381
dist_min_ci_0_5h          1.634553
cross_track_component     2.425414
log1p_growth              3.635148
fire_pressure             3.674657
dist_fit_r2_0_5h          3.884857
area_first_ha             4.721167
log_area_ratio_0_5h       6.337715
centroid_speed_m_per_h    6.959181
distance_speed            7.843248
dtype: float64

### Skewness Check – Final Decision

Current skew values are within an acceptable range for modeling.

**Key Points**
- Features already transformed with `log1p` should **not be transformed again**:
  - `log1p_area_first`
  - `log1p_growth`
  - `log_area_ratio_0_5h`
- Most remaining features have **|skew| < 5**, which is acceptable.
- Cox Survival models and tree-based models **do not require perfectly normal distributions**.

**Conclusion**
No further transformations are needed.  
The dataset is ready for modeling.

**Next Step**
Save the cleaned dataset for the modeling pipeline.

```python
df_train_cleaned.to_csv("../data/train_survival_ready.csv", index=False)

In [35]:
import joblib

joblib.dump(pt, "../models/power_transformer.pkl")


# import joblib

# pt = joblib.load("../models/power_transformer.pkl")

# df_test["dist_accel_m_per_h2"] = pt.transform(df_test[["dist_accel_m_per_h2"]])

['../models/power_transformer.pkl']

In [36]:
df_train_feature.to_csv("../data/train_feature_engineered.csv", index=False)